In [2]:
import duckdb
import pandas as pd
from pathlib import Path
import numpy as np

class DataHubReader:
    def __init__(self, db_name="silver_company.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data"  / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")
        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()
    def sql_command(self, command):
        return self.con.execute(f'{command}').df()
    def get_prices(self):
        return self.con.execute(f'Select * from observations').df()
    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [21]:
hub = DataHubReader()
data = hub.sql_command("Select * from observations")
data = data.sort_values('EntityID')
data = data[~data['ItemName'].isin(['High', 'Low', 'Open'])]
test = hub.sql_command("Select * from observations")
hub.close()


In [4]:
#Gesamtanzahl der Firmen 
total_companies= len(data['EntityCode'].unique())
total_companies

8192

In [5]:
coverage = (
    data.groupby('ItemName')['EntityCode']
    .nunique()
    .reset_index()
    .rename(columns={'EntityCode': 'company_count'})
    .sort_values('company_count', ascending=False)
)
coverage['coverage_pct'] = coverage['company_count'] / total_companies * 100
coverage.head()

,ItemName,company_count,coverage_pct
56,Close,8189,99.963379
346,Volume,8189,99.963379
355,shares_outstanding,8125,99.182129
328,TotalEquityGrossMinorityInterest,8116,99.072266
325,TotalAssets,8115,99.060059


In [6]:
#Auswahl wie viel Prozent der Unternehmen das Feature haben müssen, damit es in die Feature Liste aufgenommen wird
usable_features = set(coverage[coverage['coverage_pct'] >= 85]['ItemName'])
print(len(usable_features))
print(usable_features)

88
{'Volume', 'TaxProvision', 'shares_outstanding', 'CashCashEquivalentsAndShortTermInvestments', 'EBITDA', 'TotalEquityGrossMinorityInterest', 'AccountsReceivable', 'Close', 'StockholdersEquity', 'TotalAssets', 'DilutedAverageShares', 'GrossPPE', 'BasicAverageShares', 'OrdinarySharesNumber', 'TotalUnusualItems', 'TotalCapitalization', 'NetIncomeFromContinuingOperations', 'CommonStockEquity', 'TaxEffectOfUnusualItems', 'NetIncomeCommonStockholders', 'InvestingCashFlow', 'ChangeInWorkingCapital', 'TotalNonCurrentLiabilitiesNetMinorityInterest', 'SpecialIncomeCharges', 'CashAndCashEquivalents', 'AccountsPayable', 'CapitalExpenditure', 'RetainedEarnings', 'CurrentDebtAndCapitalLeaseObligation', 'FinancingCashFlow', 'ReconciledDepreciation', 'LongTermDebtAndCapitalLeaseObligation', 'CurrentLiabilities', 'WorkingCapital', 'FreeCashFlow', 'NetIncomeFromContinuingOperationNetMinorityInterest', 'EndCashPosition', 'NormalizedIncome', 'InterestExpense', 'NetIncome', 'TotalLiabilitiesNetMinorityI

In [7]:
#Für jeden EntityCode wird ein Set mit den vorhandenen ItemNames erstellt.
features_per_company = data.groupby('EntityCode')['ItemName'].apply(set)
#Erstellt ein DataFrame indem es quasi über alle EntityCodes iteriert und prüft ob mindestens die usable_features enthalten sind. Dann gibt es eine Liste von EntityCodes zurück für die die Prüfung True ist.
matching_companies = features_per_company[features_per_company.apply(lambda s: usable_features.issubset(s))].index.tolist()
print(f"Firmen mit mindestens diesen Features: {len(matching_companies)}")
#Hier werden aus dem Gesamtdaten DataFrame die EntityCodes extrahiert die in der matching_companies Liste enthalten sind.
comp_with_features = data[(data['EntityCode'].isin(matching_companies)) & (data['ItemName'].isin(usable_features))].copy()

Firmen mit mindestens diesen Features: 3040


In [8]:
comp_with_features

,ObservationID,EntityID,ItemID,Date,Value,Unit,EntityCode,ItemName
8984862,130540,11,297,2024-12-31,1.234100e+07,USD,0013.HK,ReconciledDepreciation
8984867,130545,11,323,2024-12-31,-1.201000e+06,USD,0013.HK,TaxEffectOfUnusualItems
8984865,130543,11,315,2024-12-31,1.129130e+08,USD,0013.HK,SellingGeneralAndAdministration
8984866,130544,11,319,2024-12-31,-4.804000e+06,USD,0013.HK,SpecialIncomeCharges
8984860,130538,11,280,2024-12-31,-1.107000e+06,USD,0013.HK,PretaxIncome
...,...,...,...,...,...,...,...,...
2415218,9470579,8191,57,2023-01-01,2.110779e+01,USD,ZWS,Close
2415217,9470578,8191,343,2022-12-31,-1.540000e+07,USD,ZWS,TotalUnusualItemsExcludingGoodwill
2415222,9470583,8191,350,2023-01-01,3.404640e+07,shares,ZWS,Volume
2415216,9470577,8191,342,2022-12-31,-1.540000e+07,USD,ZWS,TotalUnusualItems


In [9]:
#Anteil an NaN-Werten
NaN_Anteil = comp_with_features['Value'].isna().sum() / len(comp_with_features) * 100
print(NaN_Anteil)

10.166771776008652


In [10]:
min_datum = comp_with_features[~comp_with_features['ItemName'].isin(['Volume', 'Close', 'institutionsPercentHeld', 'institutionsFloatPercentHeld', 'institutionsCount', 'insidersPercentHeld'])]['Date'].min()
max_datum = comp_with_features[~comp_with_features['ItemName'].isin(['Volume', 'Close', 'institutionsPercentHeld', 'institutionsFloatPercentHeld', 'institutionsCount', 'insidersPercentHeld'])]['Date'].max()
print(min_datum)
print(max_datum)

2021-06-30 00:00:00
2026-06-07 00:00:00


In [11]:
cv = comp_with_features[comp_with_features['ItemName'].isin(['Volume', 'Close'])]
shares = comp_with_features[comp_with_features['ItemName'] == 'shares_outstanding']
funda = comp_with_features[~comp_with_features['ItemName'].isin(['Volume', 'Close', 'institutionsPercentHeld', 'institutionsFloatPercentHeld', 'institutionsCount', 'insidersPercentHeld'])]
major_holders = comp_with_features[comp_with_features['ItemName'].isin(['institutionsPercentHeld', 'institutionsFloatPercentHeld', 'institutionsCount', 'insidersPercentHeld'])]

In [12]:
group = cv.groupby(['EntityCode', 'ItemName'])
result = []
for name, g  in group:
    g = g.set_index('Date').sort_index()
    g = g[~g.index.duplicated(keep='last')]
    g = g.ffill()
    g = g.reset_index()
    result.append(g)

cv_filled = pd.concat(result, ignore_index=True)
cv_filled

,Date,ObservationID,EntityID,ItemID,Value,Unit,EntityCode,ItemName
0,2021-07-01,129625,11,57,9.092829e+00,USD,0013.HK,Close
1,2021-08-01,129630,11,57,7.541789e+00,USD,0013.HK,Close
2,2021-09-01,129635,11,57,7.328456e+00,USD,0013.HK,Close
3,2021-10-01,129640,11,57,5.946464e+00,USD,0013.HK,Close
4,2021-11-01,129645,11,57,7.197731e+00,USD,0013.HK,Close
...,...,...,...,...,...,...,...,...
341423,2026-02-01,9471473,8191,350,2.065890e+07,shares,ZWS,Volume
341424,2026-03-01,9471481,8191,350,2.126900e+07,shares,ZWS,Volume
341425,2026-04-01,9471487,8191,350,2.830550e+07,shares,ZWS,Volume
341426,2026-05-01,9471493,8191,350,1.753340e+07,shares,ZWS,Volume


In [13]:
pd.set_option('display.max_rows', 100)
def resample_to_monthly(df: pd.DataFrame) -> pd.DataFrame:
    def resample_entity(group):
        #Hier kommt ein DF mit allen Daten eines EntityCodes rein, das Datum wird als Index gesetzt, sortiert und dann der überall der letzte Wert eines Monats auf den letzten Tag eines Monats gesetzt. Wenn ein Monat 
        #keinen Wert hat, dann bekommt er NaN. Diese NaN-Werte werden dann aber durch den Forwardfill überschrieben. Das wird dann durch apply auf jede EntityCode Gruppe angewendet.
        group = group.set_index('Date').sort_index()
        monthly = group.resample('M').last().ffill()
        return monthly.reset_index()
    #Hier wird alles nach EntityCode gruppiert und die obere Funktion darauf angewendet. Group_Keys=False verhindert, dass EntityCode dann als Index bleibt.
    return (df.groupby('EntityCode', group_keys=False).apply(resample_entity).reset_index(drop=True))

shares_outstanding = resample_to_monthly(shares)

In [18]:
def extend_to_date_range(df: pd.DataFrame, min_date, max_date) -> pd.DataFrame:
    """
    Input:  DataFrame mit Spalten ['EntityCode', 'ItemName', 'Date', 'Value']
            min_date, max_date: gewünschter Zeitraum
    Output: DataFrame mit vollständiger Monatsreihe von min_date bis max_date
            pro EntityCode + ItemName Kombination
    """
    full_index = pd.date_range(start=min_date, end=max_date, freq='M')

    def extend_group(group):
        group = group.set_index('Date').sort_index()
        group = group.reindex(full_index)
        group['Value'] = group['Value'].ffill().bfill()
        group['EntityCode'] = group['EntityCode'].ffill().bfill()
        group['ItemName'] = group['ItemName'].ffill().bfill()
        return group.reset_index().rename(columns={'index': 'Date'})

    return (
        df.groupby(['EntityCode', 'ItemName'], group_keys=False)
        .apply(extend_group)
        .reset_index(drop=True)
    )
extend_to_date_range(comp_with_features, min_datum, max_datum)

ValueError: cannot reindex on an axis with duplicate labels

In [15]:
def resample_to_monthly(
    df: pd.DataFrame,
    min_date: str,
    max_date: str,
) -> pd.DataFrame:
    """
    Resamples all (EntityCode, ItemName) time series to a complete monthly
    (month-end) grid between min_date and max_date.

    - Value is NaN for filled months (no original observation)
    - All other columns (ObservationID, EntityID, ItemID, Unit, EntityCode,
      ItemName) are forward-filled from the last known row
    - Dates are always normalized to month-end (MS -> ME)

    Parameters
    ----------
    df       : Raw observations DataFrame
    min_date : Start of the target grid, e.g. '2010-01-31'
    max_date : End of the target grid,   e.g. '2024-12-31'

    Returns
    -------
    DataFrame with the same columns, monthly frequency, sorted by
    (EntityCode, ItemName, Date).
    """

    min_date = pd.Timestamp(min_date) + pd.offsets.MonthEnd(0)
    max_date = pd.Timestamp(max_date) + pd.offsets.MonthEnd(0)

    # Normalize all existing dates to month-end
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)

    # Full target grid (month-end)
    target_index = pd.date_range(min_date, max_date, freq="M")

    meta_cols = ["ObservationID", "EntityID", "ItemID", "Unit", "EntityCode", "ItemName"]

    def _expand_group(grp: pd.DataFrame) -> pd.DataFrame:
        # One row per month-end; Value becomes NaN for inserted rows
        grp = grp.set_index("Date")
        grp = grp[~grp.index.duplicated(keep="last")]
        grp = grp.reindex(target_index)

        # Forward-fill all metadata columns, but NOT Value
        grp[meta_cols] = grp[meta_cols].ffill().bfill()

        # Restore Date as column
        grp = grp.reset_index().rename(columns={"index": "Date"})
        return grp

    result = (
        df.groupby(["EntityCode", "ItemName"], sort=False)
          .apply(_expand_group)
          .reset_index(drop=True)
    )

    return result.sort_values(["EntityCode", "ItemName", "Date"]).reset_index(drop=True)

raw_data = resample_to_monthly(comp_with_features, min_datum, max_datum)

In [16]:
isna = raw_data['Value'].isna().sum()
ges = len(raw_data)

isna / ges

0.9164889770766335

In [19]:
raw_data.head(60)

,Date,ObservationID,EntityID,ItemID,Value,Unit,EntityCode,ItemName
0,2021-06-30,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
1,2021-07-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
2,2021-08-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
3,2021-09-30,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
4,2021-10-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
5,2021-11-30,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
6,2021-12-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
7,2022-01-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
8,2022-02-28,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
9,2022-03-31,129655.0,11.0,1.0,NaN,USD,0013.HK,AccountsPayable
